# PET Lab — SCU HPC — AMD MI100 GPU Training Pipeline (v45)

This notebook runs on the **SCU HPC** with MI100 GPUs (32 GB HBM2).

## What this does
1. **Verifies GPU access** (ROCm + PyTorch)
2. **Loads ESM-2 3B** on GPU — uses the MI100's 32 GB HBM2 (vs. 150 M locally)
3. **Computes ESM-2 3B embeddings** for all training proteins (GPU-accelerated)
4. **Loads all v45 feature caches** (physics, conservation, metal coordination, pLDDT, ESM log-likelihood, ESM-1v)
5. **Extracts 141-feature vectors** per mutation — full v45 feature set
6. **Trains the full 8-model ensemble** with wide-stack meta-classifier
7. **Exports model weights** for download to local backend

## Upload these files before running
| File | Source |
|------|--------|
| `4_fireprotDB_bestpH.csv` | `fireprotdb_data/fireprot_upload/csvs/` |
| `thermomutdb.json` | project root |
| `conservation_cache.pkl` | `backend/app/trained_models/` |
| `metal_coord_cache.pkl` | `backend/app/trained_models/` |
| `physics_features_cache.pkl` | `backend/app/trained_models/` |
| `plddt_pdb_cache.pkl` | `backend/app/trained_models/` |
| `esm_loglik_cache.pkl` | `backend/app/trained_models/` |
| `esm1v_ensemble_cache.pkl` | `backend/app/trained_models/` |

## Expected output
- CV Accuracy > 79.68% (current v45 baseline) thanks to ESM-2 3B embeddings
- Download `trained_models_mi300x/` → copy to `backend/app/trained_models/`

## Step 0: Verify AMD MI300X GPU

In [ ]:
!rocm-smi
print('=' * 60)

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'ROCm available (torch.cuda): {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'Number of GPUs: {torch.cuda.device_count()}')
else:
    raise RuntimeError('No GPU detected! Check ROCm installation.')

## Step 1: Install dependencies

In [ ]:
!pip install fair-esm pandas scikit-learn xgboost lightgbm catboost optuna scipy --quiet

## Step 2: Load ESM-2 3B on SCU HPC (MI100 GPU)

The MI100 has 32 GB HBM2. ESM-2 3B (~12 GB) fits comfortably and produces **much richer** embeddings than the
ESM-2 150 M model used locally. This is the primary reason to run here.

In [ ]:
import esm
import torch
import numpy as np
import time

print('Loading ESM-2 3B model on MI300X...')
t0 = time.time()
# esm2_t36_3B_UR50D = 3 billion params, 2560-dim embeddings, 36 layers
model, alphabet = esm.pretrained.esm2_t36_3B_UR50D()
batch_converter = alphabet.get_batch_converter()
model = model.eval().cuda()
print(f'Model loaded in {time.time()-t0:.1f}s')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

ESM_EMBED_DIM = 2560  # ESM-2 3B embedding dimension (vs. 640 for 150M)

## Step 3: Load training data (FireProtDB + ThermoMutDB)

In [ ]:
import pandas as pd
import json
import re
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')
AA_SET = set(AMINO_ACIDS)

def parse_mutation_code(code):
    m = re.match(r'^([A-Z])(\d+)([A-Z])$', str(code))
    if m:
        return m.group(1), int(m.group(2)), m.group(3)
    return None, None, None

# ── FireProtDB ──────────────────────────────────────────────────────────────
df_fp = pd.read_csv('4_fireprotDB_bestpH.csv')
MAX_ASA = {
    'A':129.0,'R':274.0,'N':195.0,'D':193.0,'C':167.0,'Q':225.0,'E':223.0,
    'G':104.0,'H':224.0,'I':197.0,'L':201.0,'K':236.0,'M':224.0,'F':240.0,
    'P':159.0,'S':155.0,'T':172.0,'W':285.0,'Y':263.0,'V':174.0,
}
fp_records = []
for _, row in df_fp.iterrows():
    wt  = row.get('wild_type', '')
    mut = row.get('mutation', '')
    pos = row.get('position', 0)
    ddg = row.get('ddG')
    if pd.isna(ddg) or wt not in AA_SET or mut not in AA_SET or wt == mut:
        continue
    try:
        pos = int(pos)
    except (ValueError, TypeError):
        continue
    ph_val = row.get('pH', row.get('ph', 7.0))
    try:
        ph_val = float(ph_val) if pd.notna(ph_val) else 7.0
    except (ValueError, TypeError):
        ph_val = 7.0
    fp_asa = row.get('asa')
    try:
        fp_asa = float(fp_asa) if pd.notna(fp_asa) else None
    except (ValueError, TypeError):
        fp_asa = None
    fp_ss = row.get('secondary_structure')
    fp_ss = str(fp_ss) if pd.notna(fp_ss) else None
    pdb = str(row.get('pdb_id', '')).split('|')[0]
    fp_records.append({
        'wt_aa': wt, 'position': pos, 'mut_aa': mut,
        'ddg': float(ddg), 'sequence': str(row.get('sequence', '')) if pd.notna(row.get('sequence', '')) else '',
        'protein_id': pdb, 'source': 'FireProtDB',
        'temperature_c': 25.0, 'ph': ph_val,
        'fp_asa': fp_asa, 'fp_ss': fp_ss,
    })
print(f'FireProtDB: {len(fp_records)} mutations')

# ── ThermoMutDB ─────────────────────────────────────────────────────────────
with open('thermomutdb.json') as f:
    thermo_data = json.load(f)
tm_records = []
dtm_records = []
for entry in thermo_data:
    wt, pos, mut = parse_mutation_code(entry.get('mutation_code', ''))
    if wt is None:
        continue
    pdb = entry.get('PDB_wild', '')
    temp_k = entry.get('temperature')
    try:
        temp_c = float(temp_k) - 273.15 if temp_k is not None else 37.0
    except (ValueError, TypeError):
        temp_c = 37.0
    ph_val = entry.get('ph')
    try:
        ph_val = float(ph_val) if ph_val is not None else 7.0
    except (ValueError, TypeError):
        ph_val = 7.0
    def _sf(v):
        try:
            return float(v) if v is not None else None
        except (ValueError, TypeError):
            return None
    base = {
        'wt_aa': wt, 'position': pos, 'mut_aa': mut,
        'sequence': '', 'protein_id': str(pdb), 'source': 'ThermoMutDB',
        'temperature_c': temp_c, 'ph': ph_val,
        'struct_rsa': _sf(entry.get('rsa')), 'struct_phi': _sf(entry.get('phi')),
        'struct_psi': _sf(entry.get('psi')), 'struct_depth': _sf(entry.get('ca_depth')),
    }
    ddg = entry.get('ddg')
    if ddg is not None:
        try:
            tm_records.append({**base, 'ddg': float(ddg)})
        except (ValueError, TypeError):
            pass
    dtm = entry.get('dtm')
    if dtm is not None:
        try:
            dtm_records.append({**base, 'dtm': float(dtm)})
        except (ValueError, TypeError):
            pass
print(f'ThermoMutDB: {len(tm_records)} DDG + {len(dtm_records)} ΔTm mutations')

# ── Combine, deduplicate, clip outliers ─────────────────────────────────────
all_records = fp_records + tm_records
seen = set()
unique_records = []
for r in all_records:
    key = (r['protein_id'], r['position'], r['wt_aa'], r['mut_aa'])
    if key not in seen:
        seen.add(key)
        unique_records.append(r)
print(f'After dedup: {len(unique_records)} mutations')
unique_records = [r for r in unique_records if abs(r['ddg']) <= 10.0]
print(f'After |DDG|<=10 clip: {len(unique_records)} mutations')

# ── Reverse mutation augmentation (thermodynamic antisymmetry) ───────────────
augmented = []
for r in unique_records:
    if r['wt_aa'] == r['mut_aa']:
        continue
    augmented.append({
        'wt_aa': r['mut_aa'], 'mut_aa': r['wt_aa'], 'position': r['position'],
        'ddg': -r['ddg'], 'sequence': r.get('sequence', ''),
        'protein_id': r['protein_id'], 'source': r['source'],
        'temperature_c': r.get('temperature_c', 25.0), 'ph': r.get('ph', 7.0),
        'struct_rsa': r.get('struct_rsa'), 'struct_phi': r.get('struct_phi'),
        'struct_psi': r.get('struct_psi'), 'struct_depth': r.get('struct_depth'),
        'fp_asa': r.get('fp_asa'), 'fp_ss': r.get('fp_ss'),
    })
seen2 = set((r['protein_id'], r['position'], r['wt_aa'], r['mut_aa']) for r in unique_records)
for r in augmented:
    key = (r['protein_id'], r['position'], r['wt_aa'], r['mut_aa'])
    if key not in seen2:
        seen2.add(key)
        unique_records.append(r)
print(f'After reverse-mutation augmentation: {len(unique_records)} total')

# Collect unique protein sequences for ESM-2 embedding
seq_map = {}
for r in unique_records:
    pid = r['protein_id']
    seq = r.get('sequence', '')
    if pid and seq and len(seq) > 10:
        seq_map[pid] = seq
print(f'Unique proteins with sequences: {len(seq_map)}')

## Step 4: Compute ESM-2 3B embeddings on MI300X

ESM-2 3B produces 2560-dim embeddings — 4× richer than the 640-dim ESM-2 150 M used locally.
On MI300X this completes in minutes; on CPU it would take hours.

In [ ]:
esm3b_cache = {}  # {protein_id: np.array(seq_len, 2560)}
proteins = {k: v for k, v in seq_map.items() if len(v) > 10}
print(f'Computing ESM-2 3B embeddings for {len(proteins)} proteins on MI300X...')

t0 = time.time()
for i, (uid, seq) in enumerate(proteins.items()):
    seq_trunc = seq[:1022]  # ESM-2 max length
    try:
        data = [('protein', seq_trunc)]
        _, _, tokens = batch_converter(data)
        tokens = tokens.cuda()
        with torch.no_grad():
            results = model(tokens, repr_layers=[36], return_contacts=False)
        # shape: (seq_len, 2560)
        reps = results['representations'][36][0, 1:len(seq_trunc)+1].cpu().numpy().astype(np.float32)
        esm3b_cache[uid] = reps
    except Exception as e:
        print(f'  Failed {uid} (len={len(seq)}): {e}')
        esm3b_cache[uid] = np.zeros((len(seq_trunc), ESM_EMBED_DIM), dtype=np.float32)

    if (i + 1) % 20 == 0 or (i + 1) == len(proteins):
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (len(proteins) - i - 1) / max(rate, 1e-6)
        print(f'  {i+1}/{len(proteins)} | {rate:.1f} proteins/sec | ETA {eta:.0f}s | '
              f'GPU mem {torch.cuda.memory_allocated()/1e9:.1f} GB')

print(f'\nDone! {len(esm3b_cache)} proteins embedded in {time.time()-t0:.1f}s')
print(f'Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

with open('esm3b_embeddings.pkl', 'wb') as f:
    pickle.dump(esm3b_cache, f)
print('Saved: esm3b_embeddings.pkl')

## Step 5: Load v45 feature caches

These files were pre-computed locally and uploaded to this environment.
They provide physics, conservation (PSSM), metal coordination, pLDDT, and ESM log-likelihood features.

In [ ]:
# ── Conservation (PSSM) ──────────────────────────────────────────────────────
_conservation_cache = {}
if os.path.exists('conservation_cache.pkl'):
    with open('conservation_cache.pkl', 'rb') as f:
        _conservation_cache = pickle.load(f)
    print(f'Conservation cache: {len(_conservation_cache)} proteins')
else:
    print('WARNING: conservation_cache.pkl not found — PSSM features will be zero')

# ── Metal coordination ───────────────────────────────────────────────────────
_metal_coord_cache = {}
if os.path.exists('metal_coord_cache.pkl'):
    with open('metal_coord_cache.pkl', 'rb') as f:
        _metal_coord_cache = pickle.load(f)
    print(f'Metal coordination cache: {len(_metal_coord_cache)} proteins')
else:
    print('WARNING: metal_coord_cache.pkl not found — metal features will be zero')

# ── Physics (Debye-Hückel + Born solvation) ──────────────────────────────────
_physics_cache = {}
if os.path.exists('physics_features_cache.pkl'):
    with open('physics_features_cache.pkl', 'rb') as f:
        _physics_cache = pickle.load(f)
    n_res = sum(len(v) for v in _physics_cache.values())
    print(f'Physics cache: {len(_physics_cache)} proteins, {n_res} residue positions')
else:
    print('WARNING: physics_features_cache.pkl not found — physics features will be approximated')

# ── pLDDT ────────────────────────────────────────────────────────────────────
_plddt_cache = {}
if os.path.exists('plddt_pdb_cache.pkl'):
    with open('plddt_pdb_cache.pkl', 'rb') as f:
        _plddt_cache = pickle.load(f)
    print(f'pLDDT cache: {len(_plddt_cache)} PDB entries')
else:
    print('WARNING: plddt_pdb_cache.pkl not found — pLDDT features will be zero')

# ── ESM-2 masked log-likelihood ───────────────────────────────────────────────
_esm_loglik_cache = {}
if os.path.exists('esm_loglik_cache.pkl'):
    with open('esm_loglik_cache.pkl', 'rb') as f:
        _esm_loglik_cache = pickle.load(f)
    n_pos = sum(len(v) for v in _esm_loglik_cache.values())
    print(f'ESM log-likelihood cache: {len(_esm_loglik_cache)} proteins, {n_pos} positions')
else:
    print('WARNING: esm_loglik_cache.pkl not found — ESM ΔLL features will be zero')

# ── ESM-1v evolutionary fitness ───────────────────────────────────────────────
_esm1v_cache = {}
if os.path.exists('esm1v_ensemble_cache.pkl'):
    with open('esm1v_ensemble_cache.pkl', 'rb') as f:
        _esm1v_cache = pickle.load(f)
    n_pos = sum(len(v) for v in _esm1v_cache.values())
    print(f'ESM-1v cache: {len(_esm1v_cache)} proteins, {n_pos} positions')
else:
    print('WARNING: esm1v_ensemble_cache.pkl not found — ESM-1v features will be zero')

print('\nAll caches loaded.')

## Step 6: Define full v45 feature extraction (141 features)

In [ ]:
from scipy.stats import pearsonr, spearmanr

# ── Amino acid property tables ────────────────────────────────────────────────
HYDROPHOBICITY = {
    'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,'G':-0.4,'H':-3.2,'I':4.5,
    'K':-3.9,'L':3.8,'M':1.9,'N':-3.5,'P':-1.6,'Q':-3.5,'R':-4.5,'S':-0.8,
    'T':-0.7,'V':4.2,'W':-0.9,'Y':-1.3,
}
VOLUME = {
    'A':88.6,'C':108.5,'D':111.1,'E':138.4,'F':189.9,'G':60.1,'H':153.2,
    'I':166.7,'K':168.6,'L':166.7,'M':162.9,'N':114.1,'P':112.7,'Q':143.8,
    'R':173.4,'S':89.0,'T':116.1,'V':140.0,'W':227.8,'Y':193.6,
}
CHARGE = {
    'A':0,'C':0,'D':-1,'E':-1,'F':0,'G':0,'H':0.5,'I':0,'K':1,'L':0,
    'M':0,'N':0,'P':0,'Q':0,'R':1,'S':0,'T':0,'V':0,'W':0,'Y':0,
}
FLEXIBILITY = {
    'A':0.36,'C':0.35,'D':0.51,'E':0.50,'F':0.31,'G':0.54,'H':0.32,'I':0.46,
    'K':0.47,'L':0.40,'M':0.30,'N':0.46,'P':0.51,'Q':0.49,'R':0.53,'S':0.51,
    'T':0.44,'V':0.39,'W':0.31,'Y':0.42,
}
HELIX_PROPENSITY = {
    'A':1.42,'C':0.70,'D':1.01,'E':1.51,'F':1.13,'G':0.57,'H':1.00,'I':1.08,
    'K':1.16,'L':1.21,'M':1.45,'N':0.67,'P':0.57,'Q':1.11,'R':0.98,'S':0.77,
    'T':0.83,'V':1.06,'W':1.08,'Y':0.69,
}
SHEET_PROPENSITY = {
    'A':0.83,'C':1.19,'D':0.54,'E':0.37,'F':1.38,'G':0.75,'H':0.87,'I':1.60,
    'K':0.74,'L':1.30,'M':1.05,'N':0.89,'P':0.55,'Q':1.10,'R':0.93,'S':0.75,
    'T':1.19,'V':1.70,'W':1.37,'Y':1.47,
}
MOLECULAR_WEIGHT = {
    'A':89.1,'C':121.2,'D':133.1,'E':147.1,'F':165.2,'G':75.0,'H':155.2,'I':131.2,
    'K':146.2,'L':131.2,'M':149.2,'N':132.1,'P':115.1,'Q':146.1,'R':174.2,'S':105.1,
    'T':119.1,'V':117.1,'W':204.2,'Y':181.2,
}
HBOND_DONORS    = {'A':1,'C':1,'D':1,'E':1,'F':1,'G':1,'H':2,'I':1,'K':2,'L':1,'M':1,'N':2,'P':0,'Q':2,'R':4,'S':2,'T':2,'V':1,'W':2,'Y':2}
HBOND_ACCEPTORS = {'A':1,'C':0,'D':3,'E':3,'F':0,'G':1,'H':1,'I':1,'K':1,'L':1,'M':2,'N':2,'P':1,'Q':2,'R':1,'S':2,'T':2,'V':1,'W':0,'Y':1}
TURN_PROPENSITY = {'A':0.66,'C':1.19,'D':1.46,'E':0.74,'F':0.60,'G':1.56,'H':0.95,'I':0.47,'K':1.01,'L':0.59,'M':0.60,'N':1.56,'P':1.52,'Q':0.98,'R':0.95,'S':1.43,'T':0.96,'V':0.50,'W':0.96,'Y':1.14}
POLARITY_CLASS  = {'A':0,'C':1,'D':2,'E':2,'F':0,'G':0,'H':2,'I':0,'K':2,'L':0,'M':0,'N':1,'P':0,'Q':1,'R':2,'S':1,'T':1,'V':0,'W':0,'Y':1}
SIDECHAIN_PKA   = {'A':0.0,'C':8.3,'D':3.9,'E':4.1,'F':0.0,'G':0.0,'H':6.0,'I':0.0,'K':10.5,'L':0.0,'M':0.0,'N':0.0,'P':0.0,'Q':0.0,'R':12.5,'S':0.0,'T':0.0,'V':0.0,'W':0.0,'Y':10.1}
SIDECHAIN_IS_ACID = {'C':True,'D':True,'E':True,'Y':True,'H':False,'K':False,'R':False}
ALIPHATIC_CONTRIB = {'A':1.0,'V':2.9,'I':3.9,'L':3.9}
SIZE_CLASS = {'G':0,'A':0,'S':1,'C':1,'T':1,'P':1,'D':1,'N':1,'V':1,'E':2,'Q':2,'I':2,'L':2,'M':2,'H':2,'K':2,'F':3,'R':3,'W':3,'Y':3}
DISORDER_PROPENSITY = {'A':0.06,'C':0.02,'D':0.19,'E':0.18,'F':-0.05,'G':0.17,'H':0.04,'I':-0.07,'K':0.16,'L':-0.07,'M':0.00,'N':0.14,'P':0.12,'Q':0.15,'R':0.14,'S':0.13,'T':0.07,'V':-0.06,'W':-0.05,'Y':-0.01}
BLOSUM62_DIAG = {'A':4,'R':5,'N':6,'D':6,'C':9,'Q':5,'E':5,'G':6,'H':8,'I':4,'L':4,'K':5,'M':5,'F':6,'P':7,'S':4,'T':5,'W':11,'Y':7,'V':4}

# Full BLOSUM62 matrix
BLOSUM62 = {}
_blosum_str = """
   A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
A  4 -1 -2 -2  0 -1 -1  0 -2 -1 -1 -1 -1 -2 -1  1  0 -3 -2  0
R -1  5  0 -2 -3  1  0 -2  0 -3 -2  2 -1 -3 -2 -1 -1 -3 -2 -3
N -2  0  6  1 -3  0  0  0  1 -3 -3  0 -2 -3 -2  1  0 -4 -2 -3
D -2 -2  1  6 -3  0  2 -1 -1 -3 -4 -1 -3 -3 -1  0 -1 -4 -3 -3
C  0 -3 -3 -3  9 -3 -4 -3 -3 -1 -1 -3 -1 -2 -3 -1 -1 -2 -2 -1
Q -1  1  0  0 -3  5  2 -2  0 -3 -2  1  0 -3 -1  0 -1 -2 -1 -2
E -1  0  0  2 -4  2  5 -2  0 -3 -3  1 -2 -3 -1  0 -1 -3 -2 -2
G  0 -2  0 -1 -3 -2 -2  6 -2 -4 -4 -2 -3 -3 -2  0 -2 -2 -3 -3
H -2  0  1 -1 -3  0  0 -2  8 -3 -3 -1 -2 -1 -2 -1 -2 -2  2 -3
I -1 -3 -3 -3 -1 -3 -3 -4 -3  4  2 -3  1  0 -3 -2 -1 -3 -1  3
L -1 -2 -3 -4 -1 -2 -3 -4 -3  2  4 -2  2  0 -3 -2 -1 -2 -1  1
K -1  2  0 -1 -3  1  1 -2 -1 -3 -2  5 -1 -3 -1  0 -1 -3 -2 -2
M -1 -1 -2 -3 -1  0 -2 -3 -2  1  2 -1  5  0 -2 -1 -1 -1 -1  1
F -2 -3 -3 -3 -2 -3 -3 -3 -1  0  0 -3  0  6 -4 -2 -2  1  3 -1
P -1 -2 -2 -1 -3 -1 -1 -2 -2 -3 -3 -1 -2 -4  7 -1 -1 -4 -3 -2
S  1 -1  1  0 -1  0  0  0 -1 -2 -2  0 -1 -2 -1  4  1 -3 -2 -2
T  0 -1  0 -1 -1 -1 -1 -2 -2 -1 -1 -1 -1 -2 -1  1  5 -2 -2  0
W -3 -3 -4 -4 -2 -2 -3 -2 -2 -3 -2 -3 -1  1 -4 -3 -2 11  2 -3
Y -2 -2 -2 -3 -2 -1 -2 -3  2 -1 -1 -2 -1  3 -3 -2 -2  2  7 -1
V  0 -3 -3 -3 -1 -2 -2 -3 -3  3  1 -2  1 -1 -2 -2  0 -3 -1  4
"""
_lines = [l for l in _blosum_str.strip().split('\n') if l.strip()]
_header = _lines[0].split()
for _line in _lines[1:]:
    _parts = _line.split()
    _aa1 = _parts[0]
    for _j, _aa2 in enumerate(_header):
        BLOSUM62[(_aa1, _aa2)] = int(_parts[_j + 1])

PSSM_AA_ORDER = list('ARNDCQEGHILKMFPSTWYV')
_ESM_LL_AA_ORDER = list('ACDEFGHIKLMNPQRSTVWY')
_ESM_LL_AA_TO_IDX = {aa: i for i, aa in enumerate(_ESM_LL_AA_ORDER)}
_ESM1V_AA_TO_IDX = {'A':5,'C':23,'D':13,'E':9,'F':18,'G':6,'H':21,'I':12,
                    'K':15,'L':4,'M':20,'N':17,'P':14,'Q':16,'R':10,'S':8,
                    'T':11,'V':7,'W':22,'Y':19}
PHYSIOLOGICAL_IONIC_STRENGTH = 0.15
IONIC_STRENGTH_SCALE = 0.5
_COULOMB   = 332.06
_EPS_WATER = 80.0
_EPS_PROT  = 4.0
_BORN_RADIUS = 3.5
_LAMBDA_D_DEF = 3.04 / (PHYSIOLOGICAL_IONIC_STRENGTH ** 0.5)

# ── Helper functions ─────────────────────────────────────────────────────────
def charge_at_ph(aa, ph):
    pka = SIDECHAIN_PKA.get(aa, 0.0)
    if pka == 0:
        return 0.0
    if SIDECHAIN_IS_ACID.get(aa, True):
        return -1.0 / (1.0 + 10.0 ** (pka - ph))
    return 1.0 / (1.0 + 10.0 ** (ph - pka))

def estimate_rsa(sequence, position):
    if not sequence or position < 1 or position > len(sequence):
        return 0.5
    idx = position - 1
    aa = sequence[idx]
    base = 0.5 - HYDROPHOBICITY.get(aa, 0) * 0.05
    rel_pos = idx / max(len(sequence) - 1, 1)
    if rel_pos < 0.05 or rel_pos > 0.95:
        base += 0.2
    window = sequence[max(0, idx-3):idx+4]
    avg_h = np.mean([HYDROPHOBICITY.get(a, 0) for a in window])
    base -= avg_h * 0.02
    return max(0.0, min(1.0, base))

def estimate_secondary_structure(sequence, position):
    if not sequence or position < 1 or position > len(sequence):
        return 0.33, 0.33, 0.34
    idx = position - 1
    window = sequence[max(0, idx-4):idx+5]
    h_score = np.mean([HELIX_PROPENSITY.get(a, 1.0) for a in window])
    s_score = np.mean([SHEET_PROPENSITY.get(a, 1.0) for a in window])
    total = h_score + s_score + 1.0
    return h_score / total, s_score / total, 1.0 / total

def get_conservation_features(protein_id, position, wt_aa, mut_aa):
    pssm_data = _conservation_cache.get(protein_id)
    if pssm_data is None:
        return [0.0] * 6
    pssm = pssm_data['pssm']
    info = pssm_data['info_content']
    idx = position - 1
    if idx < 0 or idx >= len(pssm):
        return [0.0] * 6
    aa_to_idx = {aa: i for i, aa in enumerate(PSSM_AA_ORDER)}
    wt_idx  = aa_to_idx.get(wt_aa)
    mut_idx = aa_to_idx.get(mut_aa)
    if wt_idx is None or mut_idx is None:
        return [0.0] * 6
    row = pssm[idx]
    pssm_wt  = float(row[wt_idx])
    pssm_mut = float(row[mut_idx])
    delta_pssm = pssm_mut - pssm_wt
    info_at_pos = float(info[idx]) if idx < len(info) else 0.0
    rank    = float(np.sum(info <= info_at_pos) / max(len(info), 1)) if len(info) > 1 else 0.5
    wt_rank = float(np.sum(row <= pssm_wt) / 20.0)
    return [pssm_wt, pssm_mut, delta_pssm, info_at_pos, rank, wt_rank]

def get_metal_features(protein_id, position):
    site_metals = _metal_coord_cache.get(protein_id, {}).get(position, set())
    if not site_metals:
        return [0.0, 0.0, 0.0, 0.0, 0.0]
    return [1.0,
            1.0 if 'CA2' in site_metals else 0.0,
            1.0 if 'ZN2' in site_metals else 0.0,
            1.0 if 'MG2' in site_metals else 0.0,
            float(len(site_metals))]

def get_physics_features(protein_id, position, wt_aa, mut_aa, ph=7.0, rsa=0.5, ionic_strength=0.15):
    def _q(aa, ph):
        pka = {'D':3.9,'E':4.1,'C':8.3,'Y':10.1,'H':6.0,'K':10.5,'R':12.5}.get(aa, 0)
        if pka == 0: return 0.0
        if aa in ('D','E','C','Y'): return -1.0/(1+10**(pka-ph))
        return +1.0/(1+10**(ph-pka))
    q_wt = _q(wt_aa, ph)
    q_mut = _q(mut_aa, ph)
    dq = q_mut - q_wt
    site_feats = _physics_cache.get(protein_id, {}).get(position) if _physics_cache else None
    if site_feats is None:
        eps_eff = _EPS_PROT + (_EPS_WATER - _EPS_PROT) * rsa
        ddg_born = -(_COULOMB/(2*_BORN_RADIUS)) * (q_mut**2 - q_wt**2) * (1/eps_eff - 1/_EPS_WATER)
        return [0.0, float(ddg_born), 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    if len(site_feats) >= 8:
        dh_scale, born_coeff, phi_site, q_local, debye_fac, elec_burial_base, contact_norm, bfactor_z = site_feats
    elif len(site_feats) >= 7:
        dh_scale, born_coeff, phi_site, q_local, debye_fac, elec_burial_base, contact_norm = site_feats
        bfactor_z = 0.0
    else:
        dh_scale, born_coeff, phi_site, q_local, debye_fac, elec_burial_base = site_feats
        contact_norm = bfactor_z = 0.0
    lam_actual  = 3.04 / max(ionic_strength, 1e-6)**0.5
    ionic_correction = lam_actual / _LAMBDA_D_DEF
    ddg_elec = float(dq * dh_scale * ionic_correction)
    eps_eff = _EPS_PROT + (_EPS_WATER - _EPS_PROT) * rsa
    ddg_born_actual = -(_COULOMB/(2*_BORN_RADIUS)) * (q_mut**2 - q_wt**2) * (1/eps_eff - 1/_EPS_WATER)
    elec_burial_actual = float(phi_site) * (1.0 - rsa)
    return [ddg_elec, float(ddg_born_actual), float(phi_site), float(q_local),
            float(debye_fac), elec_burial_actual, float(contact_norm), float(bfactor_z)]

def get_plddt_features(protein_id, position):
    scores = _plddt_cache.get(protein_id)
    if scores is None:
        return 0.0, 0.0, 0.0
    idx = int(position) - 1
    if idx < 0 or idx >= len(scores):
        return 0.0, 0.0, 0.0
    pos_score = float(scores[idx])
    window = scores[max(0, idx-5):idx+6]
    mean_score = float(np.mean(window))
    return pos_score / 100.0, mean_score / 100.0, float(pos_score < 50.0)

def extract_features(wt_aa, position, mut_aa, sequence=None, protein_id=None,
                     temperature=25.0, ph=7.0, struct_rsa=None, struct_phi=None,
                     struct_psi=None, struct_depth=None, fp_asa=None, fp_ss=None,
                     ionic_strength=None):
    if wt_aa not in AA_SET or mut_aa not in AA_SET:
        return None
    features = []
    dH = HYDROPHOBICITY.get(mut_aa,0) - HYDROPHOBICITY.get(wt_aa,0)
    dV = VOLUME.get(mut_aa,0) - VOLUME.get(wt_aa,0)
    dC = CHARGE.get(mut_aa,0) - CHARGE.get(wt_aa,0)
    dF = FLEXIBILITY.get(mut_aa,0) - FLEXIBILITY.get(wt_aa,0)
    dHelix = HELIX_PROPENSITY.get(mut_aa,1) - HELIX_PROPENSITY.get(wt_aa,1)
    dSheet = SHEET_PROPENSITY.get(mut_aa,1) - SHEET_PROPENSITY.get(wt_aa,1)
    features.extend([dH, dV, dC, dF, dHelix, dSheet])
    features.extend([abs(dH), abs(dV), abs(dC), abs(dF), abs(dHelix), abs(dSheet)])
    features.append(BLOSUM62.get((wt_aa, mut_aa), 0))
    if fp_ss is not None:
        if fp_ss in ('H','G','I'): h,s,c = 1.0,0.0,0.0
        elif fp_ss == 'E':         h,s,c = 0.0,1.0,0.0
        else:                      h,s,c = 0.0,0.0,1.0
    elif sequence:
        h,s,c = estimate_secondary_structure(sequence, position)
    else:
        h,s,c = 0.33,0.33,0.34
    features.extend([h, s, c])
    has_real_rsa_flag = 0
    if struct_rsa is not None:
        rsa = max(0.0, min(1.0, float(struct_rsa)))
        has_real_rsa_flag = 1
    elif fp_asa is not None:
        rsa = max(0.0, min(1.0, float(fp_asa) / MAX_ASA.get(wt_aa, 200.0)))
        has_real_rsa_flag = 1
    else:
        rsa = estimate_rsa(sequence, position) if sequence else 0.5
    features.append(rsa)
    if sequence and 1 <= position <= len(sequence):
        idx = position - 1
        window = sequence[max(0,idx-3):idx+4]
        local_h = np.mean([HYDROPHOBICITY.get(a,0) for a in window])
        local_c = np.mean([CHARGE.get(a,0) for a in window])
        gp_count = sum(1 for a in window if a in ('G','P'))
        rel_pos = idx / max(len(sequence)-1, 1)
        features.extend([local_h, local_c, gp_count/len(window), rel_pos])
    else:
        features.extend([0, 0, 0, 0.5])
    to_proline = 1.0 if mut_aa == 'P' and wt_aa != 'P' else 0.0
    from_proline = 1.0 if wt_aa == 'P' and mut_aa != 'P' else 0.0
    to_glycine = 1.0 if mut_aa == 'G' and wt_aa != 'G' else 0.0
    deamid_risk = 0.0
    if wt_aa in ('N','Q') and mut_aa not in ('N','Q'): deamid_risk = -1.0
    elif mut_aa in ('N','Q') and wt_aa not in ('N','Q'): deamid_risk = 1.0
    salt_bridge = 0.0
    if mut_aa in ('D','E','K','R') and wt_aa not in ('D','E','K','R'): salt_bridge = 1.0
    elif wt_aa in ('D','E','K','R') and mut_aa not in ('D','E','K','R'): salt_bridge = -1.0
    cys_change = 0.0
    if mut_aa == 'C' and wt_aa != 'C': cys_change = 1.0
    elif wt_aa == 'C' and mut_aa != 'C': cys_change = -1.0
    features.extend([to_proline, from_proline, to_glycine, deamid_risk, salt_bridge, cys_change])
    burial = 1.0 - rsa
    features.extend([abs(dH)*burial, abs(dV)*burial, abs(dC)*burial, abs(dH)*abs(dV),
                     abs(dC)*abs(dH), to_proline*burial, burial*h, burial*s, abs(dH)*h])
    aromatic_wt  = 1.0 if wt_aa  in ('F','W','Y','H') else 0.0
    aromatic_mut = 1.0 if mut_aa in ('F','W','Y','H') else 0.0
    small_aa = {'G','A','S','T','C'}
    large_aa = {'F','W','Y','R','K','H'}
    cons_wt  = BLOSUM62_DIAG.get(wt_aa, 4)
    cons_mut = BLOSUM62_DIAG.get(mut_aa, 4)
    features.extend([aromatic_wt-aromatic_mut,
                     1.0 if wt_aa in small_aa and mut_aa in large_aa else 0.0,
                     1.0 if wt_aa in large_aa and mut_aa in small_aa else 0.0,
                     cons_wt, cons_mut, cons_wt-cons_mut])
    features.extend(get_conservation_features(protein_id, position, wt_aa, mut_aa))
    features.extend([float(temperature), float(ph)])
    dMW = MOLECULAR_WEIGHT.get(mut_aa,130.0) - MOLECULAR_WEIGHT.get(wt_aa,130.0)
    dHD = float(HBOND_DONORS.get(mut_aa,1) - HBOND_DONORS.get(wt_aa,1))
    dHA = float(HBOND_ACCEPTORS.get(mut_aa,1) - HBOND_ACCEPTORS.get(wt_aa,1))
    dTurn = TURN_PROPENSITY.get(mut_aa,1.0) - TURN_PROPENSITY.get(wt_aa,1.0)
    pol_change = float(POLARITY_CLASS.get(mut_aa,0) - POLARITY_CLASS.get(wt_aa,0))
    charge_gain = 1.0 if POLARITY_CLASS.get(mut_aa,0)==2 and POLARITY_CLASS.get(wt_aa,0)!=2 else 0.0
    charge_loss = 1.0 if POLARITY_CLASS.get(wt_aa,0)==2 and POLARITY_CLASS.get(mut_aa,0)!=2 else 0.0
    pka_wt  = SIDECHAIN_PKA.get(wt_aa,0.0)
    pka_mut = SIDECHAIN_PKA.get(mut_aa,0.0)
    ion_wt  = 1.0/(1.0+10.0**(pka_wt-float(ph)))  if pka_wt>0 else 0.0
    ion_mut = 1.0/(1.0+10.0**(pka_mut-float(ph))) if pka_mut>0 else 0.0
    features.extend([dMW, dHD, dHA, dTurn, pol_change, charge_gain, charge_loss, ion_mut-ion_wt])
    dAliphatic = ALIPHATIC_CONTRIB.get(mut_aa,0.0) - ALIPHATIC_CONTRIB.get(wt_aa,0.0)
    ch_wt  = charge_at_ph(wt_aa, ph)
    ch_mut = charge_at_ph(mut_aa, ph)
    dCharge_ph = ch_mut - ch_wt
    dSizeClass = float(SIZE_CLASS.get(mut_aa,1) - SIZE_CLASS.get(wt_aa,1))
    dDisorder = DISORDER_PROPENSITY.get(mut_aa,0.0) - DISORDER_PROPENSITY.get(wt_aa,0.0)
    if sequence and 1 <= position <= len(sequence):
        idx = position - 1
        win = sequence[max(0,idx-5):idx+6]
        aa_counts = {a: win.count(a) for a in set(win)}
        n = len(win)
        entropy = -sum((c/n)*np.log2(c/n) for c in aa_counts.values() if c > 0)
    else:
        entropy = 2.0
    buried_h_wt  = 1.0 if (rsa < 0.25 and HYDROPHOBICITY.get(wt_aa,0) > 1.5) else 0.0
    buried_h_mut = 1.0 if (rsa < 0.25 and HYDROPHOBICITY.get(mut_aa,0) > 1.5) else 0.0
    if sequence and 1 <= position <= len(sequence):
        idx = position - 1
        cys_positions = [i for i,a in enumerate(sequence) if a == 'C']
        nearest_cys_dist = min(abs(idx-cp) for cp in cys_positions)/max(len(sequence),1) if cys_positions else 1.0
    else:
        nearest_cys_dist = 1.0
    features.extend([dAliphatic, dCharge_ph, dSizeClass, dDisorder, entropy,
                     buried_h_wt, buried_h_mut, abs(ch_wt), abs(ch_mut), nearest_cys_dist])
    temp_scaled = float(temperature) * 0.01
    features.extend([
        dH*dC, dH*dV, dCharge_ph*float(ph), dH*temp_scaled,
        burial*dCharge_ph, burial*dMW*0.01, abs(dH)*abs(dCharge_ph), dAliphatic*temp_scaled,
    ])
    phi_norm   = float(struct_phi)/180.0   if struct_phi   is not None else 0.0
    psi_norm   = float(struct_psi)/180.0   if struct_psi   is not None else 0.0
    depth_norm = float(struct_depth)/20.0  if struct_depth is not None else 0.0
    features.extend([phi_norm, psi_norm, depth_norm, float(has_real_rsa_flag)])
    features.extend([
        phi_norm*psi_norm, phi_norm*dH, phi_norm*temp_scaled,
        psi_norm*dH, psi_norm*temp_scaled, depth_norm*dH,
    ])
    features.extend(get_metal_features(protein_id, position))
    ionic_norm = float(ionic_strength)/IONIC_STRENGTH_SCALE if ionic_strength is not None else PHYSIOLOGICAL_IONIC_STRENGTH/IONIC_STRENGTH_SCALE
    features.append(ionic_norm)
    actual_ionic = ionic_strength if ionic_strength is not None else PHYSIOLOGICAL_IONIC_STRENGTH
    features.extend(get_physics_features(protein_id, position, wt_aa, mut_aa,
                                          ph=float(ph), rsa=rsa, ionic_strength=actual_ionic))
    pid_upper = protein_id.upper() if protein_id else ''
    delta_ll = wt_logprob = mut_logprob = 0.0
    mut_rank = 10.0
    site_entropy_ll = 0.0
    if _esm_loglik_cache:
        pos_logliks = _esm_loglik_cache.get(pid_upper, {}).get(position)
        if pos_logliks is not None:
            wi = _ESM_LL_AA_TO_IDX.get(wt_aa,-1)
            mi = _ESM_LL_AA_TO_IDX.get(mut_aa,-1)
            if wi >= 0 and mi >= 0:
                delta_ll   = float(pos_logliks[mi] - pos_logliks[wi])
                wt_logprob = float(pos_logliks[wi])
                mut_logprob= float(pos_logliks[mi])
                mut_rank   = float(np.sum(pos_logliks > pos_logliks[mi]))
            probs = np.exp(pos_logliks - np.max(pos_logliks))
            probs /= probs.sum()
            site_entropy_ll = float(-np.sum(probs * np.log(probs + 1e-10)))
    features.extend([delta_ll, wt_logprob, mut_logprob, mut_rank, site_entropy_ll])
    esm1v_delta_ll = esm1v_wt_lp = esm1v_mut_lp = 0.0
    if _esm1v_cache:
        esm1v_pos = (_esm1v_cache.get(pid_upper) or {}).get(position)
        if esm1v_pos is None:
            esm1v_pos = (_esm1v_cache.get(protein_id) or {}).get(position)
        if esm1v_pos is not None:
            wi = _ESM1V_AA_TO_IDX.get(wt_aa,-1)
            mi = _ESM1V_AA_TO_IDX.get(mut_aa,-1)
            if wi >= 0 and mi >= 0:
                esm1v_delta_ll = float(esm1v_pos[mi] - esm1v_pos[wi])
                esm1v_wt_lp   = float(esm1v_pos[wi])
                esm1v_mut_lp  = float(esm1v_pos[mi])
    features.extend([esm1v_delta_ll, esm1v_wt_lp, esm1v_mut_lp])
    return features  # 108 base features (+ ESM-2 3B PCA-32 + has_esm + 3 pLDDT = 144 total)

print('Feature extraction functions defined (108 base features).')

## Step 7: Extract features + fit PCA on ESM-2 3B embeddings

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

ESM_DIM = 32  # PCA components from 2560-dim ESM-2 3B embeddings

print(f'Extracting features for {len(unique_records)} mutations...')
base_feats_list = []
raw_esm_list    = []
valid_esm_mask  = []
y_ddg_list      = []
y_bin_list      = []
prot_list       = []
skipped = 0

for r in unique_records:
    feats = extract_features(
        r['wt_aa'], r['position'], r['mut_aa'],
        r.get('sequence',''), protein_id=r.get('protein_id'),
        temperature=r.get('temperature_c',25.0), ph=r.get('ph',7.0),
        struct_rsa=r.get('struct_rsa'), struct_phi=r.get('struct_phi'),
        struct_psi=r.get('struct_psi'), struct_depth=r.get('struct_depth'),
        fp_asa=r.get('fp_asa'), fp_ss=r.get('fp_ss'),
    )
    if feats is None:
        skipped += 1
        continue

    pid = r.get('protein_id','')
    pos = r['position']
    emb_mat = esm3b_cache.get(pid)
    if emb_mat is not None:
        idx = pos - 1
        esm_emb = emb_mat[idx] if 0 <= idx < len(emb_mat) else None
    else:
        esm_emb = None

    base_feats_list.append(feats)
    raw_esm_list.append(esm_emb)
    valid_esm_mask.append(esm_emb is not None)
    y_ddg_list.append(r['ddg'])
    y_bin_list.append(1 if r['ddg'] < 0 else 0)
    prot_list.append(pid)

print(f'  Skipped {skipped} mutations (invalid amino acids)')
n = len(base_feats_list)
esm_coverage = sum(valid_esm_mask)
print(f'  ESM-2 3B coverage: {esm_coverage}/{n} ({100*esm_coverage/max(n,1):.1f}%)')

X_base = np.array(base_feats_list, dtype=np.float32)
valid_idx = [i for i, v in enumerate(valid_esm_mask) if v]
raw_esm_valid = np.array([raw_esm_list[i] for i in valid_idx], dtype=np.float32)

print(f'  Fitting PCA({ESM_DIM}) on {len(valid_idx)} ESM-2 3B embeddings ({raw_esm_valid.shape[1]}-dim)...')
_pca = PCA(n_components=ESM_DIM, random_state=42)
_pca.fit(raw_esm_valid)
var_exp = _pca.explained_variance_ratio_.sum()
print(f'  PCA variance explained: {100*var_exp:.1f}%')
_pca_mean = _pca.transform(raw_esm_valid).mean(axis=0)

esm_features = np.tile(_pca_mean, (n, 1)).astype(np.float32)
has_esm_flag = np.zeros((n, 1), dtype=np.float32)
if len(valid_idx) > 0:
    esm_features[valid_idx] = _pca.transform(raw_esm_valid)
    has_esm_flag[valid_idx] = 1.0

X = np.concatenate([X_base, esm_features, has_esm_flag], axis=1)

# ── pLDDT features ───────────────────────────────────────────────────────────
plddt_feats = np.array([
    get_plddt_features(r.get('protein_id',''), r['position'])
    for r in unique_records[:n]
], dtype=np.float32)

# Scale, then append pLDDT
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = np.hstack([X_scaled, plddt_feats])

y_ddg = np.array(y_ddg_list)
y_bin = np.array(y_bin_list)

# Pair-constrained groups (prevents reverse-aug leakage in GroupKFold)
pair_keys = []
for r in unique_records[:n]:
    k1 = f"{r['wt_aa']}{r['position']}{r['mut_aa']}"
    k2 = f"{r['mut_aa']}{r['position']}{r['wt_aa']}"
    pair_keys.append(min(k1, k2))
unique_pair_keys = {k: i for i, k in enumerate(sorted(set(pair_keys)))}
train_groups = np.array([unique_pair_keys[k] for k in pair_keys])

print(f'\nFinal feature matrix: {X_scaled.shape[0]} samples × {X_scaled.shape[1]} features')
print(f'Stabilizing (DDG<0): {y_bin.sum()}  |  Destabilizing: {len(y_bin)-y_bin.sum()}')

## Step 8: Train 8-model ensemble

In [ ]:
from sklearn.ensemble import (GradientBoostingRegressor, HistGradientBoostingRegressor,
                               RandomForestRegressor, ExtraTreesRegressor)
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Pre-tuned XGBoost params (Optuna v26, 100 trials)
best_xgb = {
    'n_estimators':589, 'max_depth':9, 'learning_rate':0.016857723394967876,
    'subsample':0.9490003831060219, 'colsample_bytree':0.7911621808522555,
    'colsample_bylevel':0.6369503663889663, 'min_child_weight':4,
    'reg_alpha':0.36448061196314285, 'reg_lambda':0.18450540313606173,
    'gamma':0.578228192086447,
}
# Pre-tuned LightGBM params (Optuna v26, 100 trials)
best_lgbm = {
    'n_estimators':1393, 'max_depth':10, 'learning_rate':0.020643460431513237,
    'num_leaves':199, 'subsample':0.9835482825580907,
    'colsample_bytree':0.5029009084366727, 'min_child_samples':6,
    'reg_alpha':0.10367941150136066, 'reg_lambda':0.41629158663921856,
    'min_split_gain':0.06831586173554616,
}
# Pre-tuned CatBoost params (Optuna v24, 50 trials)
best_cb = {
    'iterations':966, 'depth':8, 'learning_rate':0.02904505270290301,
    'subsample':0.9423044862563902, 'colsample_bylevel':0.9268902334716056,
    'l2_leaf_reg':2.651194230102684,
}

print('Training GradientBoostingRegressor...')
gb_reg = GradientBoostingRegressor(
    n_estimators=1000, max_depth=5, learning_rate=0.03, subsample=0.8,
    min_samples_leaf=6, min_samples_split=12, max_features='sqrt',
    loss='huber', alpha=0.9, random_state=42)
gb_reg.fit(X_scaled, y_ddg)

print('Training XGBoostRegressor (pre-tuned)...')
xgb_reg = XGBRegressor(**best_xgb, random_state=42, verbosity=0, n_jobs=-1,
                        tree_method='hist', device='cuda')
xgb_reg.fit(X_scaled, y_ddg)

print('Training LightGBMRegressor (pre-tuned)...')
lgbm_reg = LGBMRegressor(**best_lgbm, n_jobs=-1, random_state=42, verbosity=-1)
lgbm_reg.fit(X_scaled, y_ddg)

print('Training CatBoostRegressor (pre-tuned)...')
cb_reg = CatBoostRegressor(**best_cb, loss_function='MAE', random_seed=42, verbose=0)
cb_reg.fit(X_scaled, y_ddg)

print('Tuning HistGradientBoosting (Optuna, 50 trials on 4k subsample)...')
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error
rng_hgb = np.random.default_rng(456)
hgb_tune_idx = rng_hgb.choice(len(X_scaled), size=4000, replace=False)
X_hgb = X_scaled[hgb_tune_idx]
y_hgb = y_ddg[hgb_tune_idx]
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)
def hgb_obj(trial):
    p = {
        'max_iter':         trial.suggest_int('max_iter', 200, 1000),
        'max_depth':        trial.suggest_int('max_depth', 3, 7),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 50),
        'l2_regularization':trial.suggest_float('l2_regularization', 1e-4, 1.0, log=True),
        'max_leaf_nodes':   trial.suggest_int('max_leaf_nodes', 15, 63),
        'random_state': 42, 'loss': 'absolute_error',
    }
    preds = cross_val_predict(HistGradientBoostingRegressor(**p), X_hgb, y_hgb, cv=kf5)
    return mean_absolute_error(y_hgb, preds)
hgb_study = optuna.create_study(direction='minimize')
hgb_study.optimize(hgb_obj, n_trials=50, show_progress_bar=False)
hgb_reg = HistGradientBoostingRegressor(**hgb_study.best_params, random_state=42, loss='absolute_error')
hgb_reg.fit(X_scaled, y_ddg)
print(f'  Best HGB MAE: {hgb_study.best_value:.4f}')

print('Training MLPRegressor...')
mlp_reg = MLPRegressor(hidden_layer_sizes=(512,256,128,64), activation='relu', solver='adam',
                        learning_rate_init=0.0005, max_iter=800, early_stopping=True,
                        validation_fraction=0.1, n_iter_no_change=30, random_state=42,
                        alpha=0.005, batch_size=256)
mlp_reg.fit(X_scaled, y_ddg)

print('Training RandomForestRegressor (1200 trees)...')
rf_reg = RandomForestRegressor(n_estimators=1200, max_depth=None, min_samples_leaf=4,
                                max_features='sqrt', n_jobs=-1, random_state=42)
rf_reg.fit(X_scaled, y_ddg)

print('Training ExtraTreesRegressor (1200 trees)...')
et_reg = ExtraTreesRegressor(n_estimators=1200, max_depth=None, min_samples_leaf=4,
                              max_features='sqrt', n_jobs=-1, random_state=42)
et_reg.fit(X_scaled, y_ddg)

models = [
    ('GradientBoosting',     gb_reg),
    ('XGBoost',              xgb_reg),
    ('LightGBM',             lgbm_reg),
    ('CatBoost',             cb_reg),
    ('HistGradientBoosting', hgb_reg),
    ('MLP',                  mlp_reg),
    ('RandomForest',         rf_reg),
    ('ExtraTrees',           et_reg),
]
print(f'\nAll {len(models)} models trained.')

## Step 9: Cross-validation + wide-stack meta-classifier

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier as CatBoostCLF
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

kf10 = GroupKFold(n_splits=10)
kf5  = GroupKFold(n_splits=5)

print('Running 10-fold GroupKFold CV on each regressor...')
cv_preds_all = {}
for name, m in models:
    cv_pred = cross_val_predict(m, X_scaled, y_ddg, cv=kf10, groups=train_groups)
    cv_preds_all[name] = cv_pred
    pr, _ = pearsonr(cv_pred, y_ddg)
    mae_v = mean_absolute_error(y_ddg, cv_pred)
    acc_v = accuracy_score(y_bin, (cv_pred < 0).astype(int))
    print(f'  {name:22s}  MAE={mae_v:.4f}  Pearson={pr:.4f}  Acc={acc_v:.4f}')

cv_ens_pred = np.mean([cv_preds_all[n] for n, _ in models], axis=0)
cv_mae = mean_absolute_error(y_ddg, cv_ens_pred)
cv_pearson, _ = pearsonr(cv_ens_pred, y_ddg)
cv_acc = accuracy_score(y_bin, (cv_ens_pred < 0).astype(int))
print(f'  {"ENSEMBLE (avg)":22s}  MAE={cv_mae:.4f}  Pearson={cv_pearson:.4f}  Acc={cv_acc:.4f}')

# ── Classifier OOF probs ──────────────────────────────────────────────────────
print('\nComputing classifier OOF probs...')
xgb_clf_base = XGBClassifier(
    n_estimators=808, max_depth=10, learning_rate=0.022895020095225958,
    subsample=0.9343533494381469, colsample_bytree=0.6053432554418391,
    colsample_bylevel=0.8949853302137403, min_child_weight=1,
    reg_alpha=1.4169700381674384, reg_lambda=0.7647093270905143,
    gamma=0.10617106081801003, eval_metric='logloss', tree_method='hist',
    random_state=42, verbosity=0, n_jobs=-1)
lgbm_clf_base = LGBMClassifier(
    n_estimators=310, max_depth=12, learning_rate=0.014486652907891896,
    num_leaves=239, subsample=0.6404629307149422,
    colsample_bytree=0.5693373118449812, min_child_samples=7,
    reg_alpha=0.0611588834567198, reg_lambda=0.00027844727233982503,
    random_state=42, verbosity=-1, n_jobs=-1)
cb_clf  = CatBoostCLF(iterations=600, depth=7, learning_rate=0.03, subsample=0.8,
                       l2_leaf_reg=3.0, random_seed=42, verbose=0)
rf_clf  = RandomForestClassifier(n_estimators=800, max_depth=None, min_samples_leaf=3,
                                  max_features='sqrt', random_state=42, n_jobs=-1)

clf_oof_probs = {}
for clf_name, clf_model in [('XGB_clf', xgb_clf_base), ('LGBM_clf', lgbm_clf_base),
                              ('CB_clf', cb_clf), ('RF_clf', rf_clf)]:
    oof_p = cross_val_predict(clf_model, X_scaled, y_bin, cv=kf10,
                               method='predict_proba', groups=train_groups)[:, 1]
    clf_oof_probs[clf_name] = oof_p
    print(f'  {clf_name:20s} OOF Acc={accuracy_score(y_bin,(oof_p>0.5).astype(int)):.4f}')

# Optuna-tuned XGB classifier (100 trials)
print('  Optuna-tuning XGBClassifier (100 trials)...')
def xgb_opt_obj(trial):
    p = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth':    trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample':    trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 5.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
    }
    clf = XGBClassifier(**p, eval_metric='logloss', tree_method='hist',
                         random_state=42, verbosity=0, n_jobs=4)
    aucs = []
    for tr_idx, va_idx in GroupKFold(n_splits=5).split(X_scaled, y_bin, groups=train_groups):
        clf.fit(X_scaled[tr_idx], y_bin[tr_idx])
        prob = clf.predict_proba(X_scaled[va_idx])[:,1]
        aucs.append(roc_auc_score(y_bin[va_idx], prob))
    return np.mean(aucs)
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(xgb_opt_obj, n_trials=100, show_progress_bar=False)
xgb_opt_clf = XGBClassifier(**study_xgb.best_params, eval_metric='logloss', tree_method='hist',
                              random_state=42, verbosity=0, n_jobs=4)
xgb_opt_oof = cross_val_predict(xgb_opt_clf, X_scaled, y_bin, cv=kf10,
                                  method='predict_proba', groups=train_groups)[:,1]
clf_oof_probs['XGB_opt'] = xgb_opt_oof
print(f'  XGB_opt Acc={accuracy_score(y_bin,(xgb_opt_oof>0.5).astype(int)):.4f}  AUC={study_xgb.best_value:.4f}')

# Optuna-tuned LGBM classifier (100 trials)
print('  Optuna-tuning LGBMClassifier (100 trials)...')
def lgbm_opt_obj(trial):
    p = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth':    trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample':    trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 5.0, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
    }
    clf = LGBMClassifier(**p, random_state=42, verbosity=-1, n_jobs=4)
    aucs = []
    for tr_idx, va_idx in GroupKFold(n_splits=5).split(X_scaled, y_bin, groups=train_groups):
        clf.fit(X_scaled[tr_idx], y_bin[tr_idx])
        prob = clf.predict_proba(X_scaled[va_idx])[:,1]
        aucs.append(roc_auc_score(y_bin[va_idx], prob))
    return np.mean(aucs)
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(lgbm_opt_obj, n_trials=100, show_progress_bar=False)
lgbm_opt_clf = LGBMClassifier(**study_lgbm.best_params, random_state=42, verbosity=-1, n_jobs=4)
lgbm_opt_oof = cross_val_predict(lgbm_opt_clf, X_scaled, y_bin, cv=kf10,
                                   method='predict_proba', groups=train_groups)[:,1]
clf_oof_probs['LGBM_opt'] = lgbm_opt_oof
print(f'  LGBM_opt Acc={accuracy_score(y_bin,(lgbm_opt_oof>0.5).astype(int)):.4f}  AUC={study_lgbm.best_value:.4f}')

# ── Build wide stack ──────────────────────────────────────────────────────────
all_reg_keys  = [n for n, _ in models]
reg_oof_mat   = np.column_stack([cv_preds_all[k] for k in all_reg_keys])
clf_oof_mat6  = np.column_stack([clf_oof_probs[n] for n in ['XGB_clf','LGBM_clf','CB_clf','RF_clf','XGB_opt','LGBM_opt']])
cv_preds_all['XGB_clf_prob']  = clf_oof_probs['XGB_clf']
cv_preds_all['LGBM_clf_prob'] = clf_oof_probs['LGBM_clf']
oof_stack = np.column_stack([cv_preds_all[n] for n in all_reg_keys + ['XGB_clf_prob','LGBM_clf_prob']])

print('\nComputing mutual information for top-30 feature selection...')
mi_scores = mutual_info_classif(X_scaled, y_bin, random_state=42)
top_feat_idx  = np.argsort(mi_scores)[-30:]
top10_feat_idx = top_feat_idx[-10:]
raw_top_mi  = X_scaled[:, top_feat_idx]
raw_top10   = X_scaled[:, top10_feat_idx]
poly_transformer = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_feats_10 = poly_transformer.fit_transform(raw_top10)[:, 10:]
clf_oof_mat2 = np.column_stack([clf_oof_probs['CB_clf'], clf_oof_probs['RF_clf']])
stack_A_poly = np.hstack([clf_oof_mat2, oof_stack, raw_top_mi, poly_feats_10])
stack_wide   = np.hstack([clf_oof_mat6, reg_oof_mat, raw_top_mi])
stack_A      = np.hstack([clf_oof_mat2, oof_stack, raw_top_mi])
stack_B      = np.hstack([clf_oof_mat2, raw_top_mi])

# LR sweep on stack_A_poly
print('\nLR sweep on stack_A_poly (5-fold GroupKFold)...')
lr_probs_by_C = {}
for C_val in [0.01, 0.1, 1.0, 10.0, 100.0]:
    probs = cross_val_predict(LogisticRegression(C=C_val, max_iter=5000, random_state=42),
                               stack_A_poly, y_bin, cv=kf5, method='predict_proba',
                               groups=train_groups)[:, 1]
    acc = accuracy_score(y_bin, (probs > 0.5).astype(int))
    lr_probs_by_C[f'stackPoly_C{C_val}'] = probs
    print(f'  LR(C={C_val:7.2f}) stack_Poly  Acc={acc:.4f}')

# XGB meta
xgb_meta_clf = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                               eval_metric='logloss', tree_method='hist',
                               random_state=42, verbosity=0, n_jobs=-1)
xgb_meta_probs = cross_val_predict(xgb_meta_clf, stack_A_poly, y_bin, cv=kf5,
                                    method='predict_proba', groups=train_groups)[:, 1]
lr_probs_by_C['XGB_meta_poly'] = xgb_meta_probs

# Ridge meta (for regression reporting)
ridge_meta = Ridge(alpha=1.0)
ridge_preds_cv = cross_val_predict(ridge_meta, stack_A_poly, y_ddg, cv=kf5, groups=train_groups)
meta_mae = mean_absolute_error(y_ddg, ridge_preds_cv)
meta_pearson, _ = pearsonr(ridge_preds_cv, y_ddg)

# Youden threshold on regressor ensemble
fpr, tpr, thr_roc = roc_curve(y_bin, -cv_ens_pred)
optimal_threshold = -thr_roc[np.argmax(tpr - fpr)]
reg_binary = (cv_ens_pred < optimal_threshold).astype(int)
reg_acc = accuracy_score(y_bin, reg_binary)

# Pick best classifier
best_clf_acc = reg_acc
best_clf_name = 'regressor_ensemble'
best_clf_probs = cv_ens_pred
best_clf_thr = optimal_threshold
for clf_name, probs in lr_probs_by_C.items():
    for thr in np.linspace(0.30, 0.70, 141):
        acc_t = accuracy_score(y_bin, (probs > thr).astype(int))
        if acc_t > best_clf_acc:
            best_clf_acc = acc_t
            best_clf_thr = thr
            best_clf_name = clf_name
            best_clf_probs = probs

meta_acc = best_clf_acc
print(f'\nBest meta: {best_clf_name}  Acc={best_clf_acc:.4f}  thr={best_clf_thr:.3f}')
print(f'CV summary: MAE={meta_mae:.4f}  Pearson={meta_pearson:.4f}  Acc={meta_acc:.4f}')

## Step 10: Refit on full data + train ΔTm regressor

In [ ]:
# Refit all models on full training data
print('Refitting all base models on full data...')
for name, m in models:
    m.fit(X_scaled, y_ddg)
    print(f'  {name} refitted.')

# Refit classifiers
for clf_name, clf_model in [('CB_clf', cb_clf), ('RF_clf', rf_clf),
                              ('XGB_opt', xgb_opt_clf), ('LGBM_opt', lgbm_opt_clf)]:
    clf_model.fit(X_scaled, y_bin)
    print(f'  {clf_name} refitted.')
xgb_clf_base.fit(X_scaled, y_bin)
lgbm_clf_base.fit(X_scaled, y_bin)

# Refit meta-learner on full stack
meta_clf_final = LogisticRegression(C=10.0, max_iter=5000, random_state=42)
meta_clf_final.fit(stack_A_poly, y_bin)
ridge_meta.fit(stack_A_poly, y_ddg)

# ── ΔTm regressor ────────────────────────────────────────────────────────────
print('\nTraining ΔTm regressor on ThermoMutDB...')
from sklearn.model_selection import KFold
dtm_X_base, dtm_esm_raw, dtm_valid_esm, dtm_y = [], [], [], []
for r in dtm_records:
    feats = extract_features(
        r['wt_aa'], r['position'], r['mut_aa'],
        r.get('sequence',''), protein_id=r.get('protein_id'),
        temperature=r.get('temperature_c',37.0), ph=r.get('ph',7.0),
        struct_rsa=r.get('struct_rsa'), struct_phi=r.get('struct_phi'),
        struct_psi=r.get('struct_psi'), struct_depth=r.get('struct_depth'),
    )
    if feats is None:
        continue
    pid = r.get('protein_id','')
    pos = r['position']
    emb_mat = esm3b_cache.get(pid)
    if emb_mat is not None:
        idx = pos - 1
        esm_emb = emb_mat[idx] if 0 <= idx < len(emb_mat) else None
    else:
        esm_emb = None
    dtm_X_base.append(feats)
    dtm_esm_raw.append(esm_emb)
    dtm_valid_esm.append(esm_emb is not None)
    dtm_y.append(r['dtm'])

dtm_base_arr = np.array(dtm_X_base, dtype=np.float32)
n_dtm = len(dtm_X_base)
dtm_esm_feats = np.tile(_pca_mean, (n_dtm, 1)).astype(np.float32)
dtm_has_esm   = np.zeros((n_dtm, 1), dtype=np.float32)
dtm_valid_idx = [i for i, v in enumerate(dtm_valid_esm) if v]
if dtm_valid_idx:
    dtm_raw_valid = np.array([dtm_esm_raw[i] for i in dtm_valid_idx], dtype=np.float32)
    dtm_esm_feats[dtm_valid_idx] = _pca.transform(dtm_raw_valid)
    dtm_has_esm[dtm_valid_idx] = 1.0
dtm_X = np.concatenate([dtm_base_arr, dtm_esm_feats, dtm_has_esm], axis=1)
dtm_plddt = np.array([get_plddt_features(r.get('protein_id',''), r['position'])
                       for r in dtm_records[:n_dtm]], dtype=np.float32)
dtm_X_scaled = scaler.transform(dtm_X)
dtm_X_scaled = np.hstack([dtm_X_scaled, dtm_plddt])
dtm_y_arr = np.array(dtm_y)
print(f'  ΔTm training set: {n_dtm} mutations')

dtm_gb = GradientBoostingRegressor(n_estimators=400, max_depth=4, learning_rate=0.05,
                                    subsample=0.8, min_samples_leaf=10, max_features='sqrt',
                                    loss='huber', alpha=0.9, random_state=42)
dtm_gb.fit(dtm_X_scaled, dtm_y_arr)
dtm_xgb = XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
                         colsample_bytree=0.8, min_child_weight=10, reg_alpha=0.1,
                         reg_lambda=1.0, random_state=42, verbosity=0, tree_method='hist', device='cuda')
dtm_xgb.fit(dtm_X_scaled, dtm_y_arr)

kf5_plain = KFold(n_splits=5, shuffle=True, random_state=42)
dtm_cv_gb  = cross_val_predict(dtm_gb,  dtm_X_scaled, dtm_y_arr, cv=kf5_plain)
dtm_cv_xgb = cross_val_predict(dtm_xgb, dtm_X_scaled, dtm_y_arr, cv=kf5_plain)
dtm_cv_ens = (dtm_cv_gb + dtm_cv_xgb) / 2.0
dtm_mae = mean_absolute_error(dtm_y_arr, dtm_cv_ens)
dtm_pearson, _ = pearsonr(dtm_cv_ens, dtm_y_arr)
print(f'  ΔTm CV: MAE={dtm_mae:.3f}°C  Pearson={dtm_pearson:.4f}')
dtm_regressor = {'models': [('GradientBoosting', dtm_gb), ('XGBoost', dtm_xgb)], 'weights': [0.5, 0.5]}
print('Done.')

## Step 11: Export model files

Download `trained_models_mi300x/` and copy its contents to `backend/app/trained_models/` on your local machine.

In [ ]:
OUTPUT_DIR = 'trained_models_mi300x'
os.makedirs(OUTPUT_DIR, exist_ok=True)

ensemble = {
    'models': [(name, model) for name, model in models],
    'weights': [1.0/len(models)] * len(models),
    'meta_learner': ridge_meta,
    'use_stacking': True,
    'optimal_threshold': float(optimal_threshold),
    'meta_type': 'Ridge',
    'clf_models': [('CB_clf', cb_clf), ('RF_clf', rf_clf)],
    'meta_clf': meta_clf_final,
    'meta_clf_threshold': float(best_clf_thr),
    'use_clf_meta': best_clf_acc > reg_acc,
    'top_feat_idx': top_feat_idx.tolist(),
    'top10_feat_idx': top10_feat_idx.tolist(),
    'poly_transformer': poly_transformer,
    'meta_uses_poly': True,
}

with open(f'{OUTPUT_DIR}/mutation_regressor.pkl', 'wb') as f:
    pickle.dump(ensemble, f)
with open(f'{OUTPUT_DIR}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(f'{OUTPUT_DIR}/deltaTm_regressor.pkl', 'wb') as f:
    pickle.dump(dtm_regressor, f)

esm_pca_data = {'pca': _pca, 'pca_mean': _pca_mean, 'esm_dim': ESM_DIM}
with open(f'{OUTPUT_DIR}/esm_pca.pkl', 'wb') as f:
    pickle.dump(esm_pca_data, f)

# Copy cache files that inference needs
import shutil
for fname in ['conservation_cache.pkl', 'metal_coord_cache.pkl',
               'physics_features_cache.pkl', 'plddt_pdb_cache.pkl',
               'esm_loglik_cache.pkl', 'esm1v_ensemble_cache.pkl']:
    if os.path.exists(fname):
        shutil.copy(fname, f'{OUTPUT_DIR}/{fname}')

meta = {
    'model_type': 'v45-mi300x: 8-model ensemble + wide-stack + ESM-2 3B PCA-32 (2560-dim)',
    'esm_model': 'ESM-2 3B (esm2_t36_3B_UR50D, 2560-dim)',
    'gpu': 'AMD Instinct MI300X',
    'prediction_target': 'DDG (kcal/mol)',
    'n_models': len(models),
    'n_features': int(X_scaled.shape[1]),
    'training_samples': int(X_scaled.shape[0]),
    'cv_mae': round(float(meta_mae), 4),
    'cv_pearson': round(float(meta_pearson), 4),
    'cv_accuracy': round(float(meta_acc), 4),
    'baseline_accuracy_v45': 0.7968,
    'synthetic_data': False,
}
with open(f'{OUTPUT_DIR}/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Exported files:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{fname}')
    print(f'  {fname}: {size/1e6:.1f} MB')

print(f'\n==== RESULTS ====')
print(f'CV Accuracy : {meta_acc:.4f}  (baseline v45: 0.7968)')
print(f'CV MAE      : {meta_mae:.4f} kcal/mol')
print(f'CV Pearson  : {meta_pearson:.4f}')
print(f'ESM model   : ESM-2 3B (2560-dim embeddings)')
print(f"\nDownload '{OUTPUT_DIR}/' and copy contents to backend/app/trained_models/")